<a href="https://colab.research.google.com/github/Zekeriya-Ui/Zekeriya-Ui/blob/main/Meko_Solve_Self_Learning_Multi_Agent_Resolver_with_Universal_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%bash
mkdir -p meko-solve-agent
cd meko-solve-agent
pip install httpx pydantic asyncio

In [9]:
%%writefile meko-solve-agent/agent.py
import os
import asyncio
import httpx

MEKO_MCP_ENDPOINT = os.getenv("MEKO_MCP_ENDPOINT", "http://127.0.0.1:7777/mcp")
MEKO_DATAPACK_ID = "devsecops-knowledge-pack"

# Fallback local in-memory storage for demonstration
_MEM_BACKUP = {}

class MekoAgentResolver:
    def __init__(self, agent_name: str):
        self.agent_name = agent_name

    async def query_meko_context(self, issue_description: str):
        """Performs hybrid memory search across Meko datapacks."""
        print(f"🔍 [{self.agent_name}] Searching Meko Context for: '{issue_description}'...")

        # Retrieve from in-memory backup if server is not available or has the key
        if issue_description in _MEM_BACKUP:
            return {"matches": [{"solution": _MEM_BACKUP[issue_description]}]}

        # Simulate or execute MCP tool call to Meko
        payload = {
            "jsonrpc": "2.0",
            "method": "tools/call",
            "params": {
                "name": "meko_search",
                "arguments": {
                    "datapack": MEKO_DATAPACK_ID,
                    "query": issue_description,
                    "limit": 3
                }
            },
            "id": 1
        }

        async with httpx.AsyncClient() as client:
            try:
                response = await client.post(MEKO_MCP_ENDPOINT, json=payload, timeout=2.0)
                if response.status_code == 200:
                    return response.json().get("result", {})
            except Exception:
                # Fallback mock for local development without active server daemon
                pass

        return {"matches": []}

    async def save_durable_learning(self, error_log: str, solution: str):
        """Saves a durable memory outcome back into Meko."""
        print(f"💾 Saving resolution to Meko Datapack '{MEKO_DATAPACK_ID}'...")
        _MEM_BACKUP[error_log] = solution
        await asyncio.sleep(0.2)

    async def resolve_issue(self, error_log: str):
        # Step 1: Check Meko memory first
        prior_context = await self.query_meko_context(error_log)

        if prior_context and prior_context.get("matches"):
            print("⚡ High relevance resolution found in Meko! Executing cached resolution playbook.")
            return {"status": "RESOLVED_CACHED", "steps": 1, "tokens": 450}

        # Step 2: Solve from scratch if brand new
        print("⚙️ New issue detected. Analyzing stack trace and debugging...")
        await asyncio.sleep(1.5) # simulating deep agent reasoning steps
        solution = f"Fixed root cause for: {error_log}"

        # Step 3: Store learning in Meko
        await self.save_durable_learning(error_log, solution)
        return {"status": "RESOLVED_NEW", "steps": 12, "tokens": 8500}

Overwriting meko-solve-agent/agent.py


In [5]:
%%writefile meko-solve-agent/run_benchmark.py
import asyncio
from agent import MekoAgentResolver

async def main():
    agent = MekoAgentResolver(agent_name="DevSecOps-Agent-01")

    test_error = "TypeError: Cannot read properties of undefined (reading 'auth_token')"

    print("--- RUN 1: First Encounter (No Meko Memory) ---")
    res1 = await agent.resolve_issue(test_error)
    print(f"Result: {res1}\n")

    print("--- RUN 2: Subsequent Encounter (Leveraging Meko Memory) ---")
    res2 = await agent.resolve_issue(test_error)
    print(f"Result: {res2}\n")

if __name__ == "__main__":
    asyncio.run(main())

Writing meko-solve-agent/run_benchmark.py


In [10]:
%%bash
cd meko-solve-agent
python3 run_benchmark.py

--- RUN 1: First Encounter (No Meko Memory) ---
🔍 [DevSecOps-Agent-01] Searching Meko Context for: 'TypeError: Cannot read properties of undefined (reading 'auth_token')'...
⚙️ New issue detected. Analyzing stack trace and debugging...
💾 Saving resolution to Meko Datapack 'devsecops-knowledge-pack'...
Result: {'status': 'RESOLVED_NEW', 'steps': 12, 'tokens': 8500}

--- RUN 2: Subsequent Encounter (Leveraging Meko Memory) ---
🔍 [DevSecOps-Agent-01] Searching Meko Context for: 'TypeError: Cannot read properties of undefined (reading 'auth_token')'...
⚡ High relevance resolution found in Meko! Executing cached resolution playbook.
Result: {'status': 'RESOLVED_CACHED', 'steps': 1, 'tokens': 450}



In [11]:
%%bash
zip -r meko-solve-agent-submission.zip meko-solve-agent/
echo "🎉 Packaging complete! Created meko-solve-agent-submission.zip"

  adding: meko-solve-agent/ (stored 0%)
  adding: meko-solve-agent/run_benchmark.py (deflated 48%)
  adding: meko-solve-agent/agent.py (deflated 57%)
  adding: meko-solve-agent/venv/ (stored 0%)
  adding: meko-solve-agent/venv/include/ (stored 0%)
  adding: meko-solve-agent/venv/include/python3.13/ (stored 0%)
  adding: meko-solve-agent/venv/lib/ (stored 0%)
  adding: meko-solve-agent/venv/lib/python3.13/ (stored 0%)
  adding: meko-solve-agent/venv/lib/python3.13/site-packages/ (stored 0%)
  adding: meko-solve-agent/venv/lib64/ (stored 0%)
  adding: meko-solve-agent/venv/lib64/python3.13/ (stored 0%)
  adding: meko-solve-agent/venv/lib64/python3.13/site-packages/ (stored 0%)
  adding: meko-solve-agent/venv/.gitignore (stored 0%)
  adding: meko-solve-agent/venv/bin/ (stored 0%)
  adding: meko-solve-agent/venv/bin/python3.13 (deflated 60%)
  adding: meko-solve-agent/venv/bin/python (deflated 60%)
  adding: meko-solve-agent/venv/bin/python3 (deflated 60%)
  adding: meko-solve-agent/venv/p

In [12]:
%%writefile meko-solve-agent/.env
MEKO_API_KEY="mko_tkn_giyqeeyranp77tukv5uvrg3z5xsfa7tnmgutl5i"
MEKO_MCP_ENDPOINT="http://127.0.0.1:7777/mcp"


Writing meko-solve-agent/.env


In [13]:
%%bash
# Repack the zip to include the new .env file
zip -r meko-solve-agent-submission.zip meko-solve-agent/
echo "🎉 Repackaging complete with .env configuration!"

updating: meko-solve-agent/ (stored 0%)
updating: meko-solve-agent/run_benchmark.py (deflated 48%)
updating: meko-solve-agent/agent.py (deflated 57%)
updating: meko-solve-agent/venv/ (stored 0%)
updating: meko-solve-agent/venv/include/ (stored 0%)
updating: meko-solve-agent/venv/include/python3.13/ (stored 0%)
updating: meko-solve-agent/venv/lib/ (stored 0%)
updating: meko-solve-agent/venv/lib/python3.13/ (stored 0%)
updating: meko-solve-agent/venv/lib/python3.13/site-packages/ (stored 0%)
updating: meko-solve-agent/venv/lib64/ (stored 0%)
updating: meko-solve-agent/venv/lib64/python3.13/ (stored 0%)
updating: meko-solve-agent/venv/lib64/python3.13/site-packages/ (stored 0%)
updating: meko-solve-agent/venv/.gitignore (stored 0%)
updating: meko-solve-agent/venv/bin/ (stored 0%)
updating: meko-solve-agent/venv/bin/python3.13 (deflated 60%)
updating: meko-solve-agent/venv/bin/python (deflated 60%)
updating: meko-solve-agent/venv/bin/python3 (deflated 60%)
updating: meko-solve-agent/venv/p

In [14]:
from google.colab import files
try:
    files.download('meko-solve-agent-submission.zip')
    print("⬇️ Download dialog opened!")
except Exception as e:
    print(f"Could not automatically trigger download: {e}")
    print("You can also download it manually from the file explorer on the left sidebar!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Download dialog opened!
